In [2]:
# ======================================================================
# CONFIG
# ======================================================================
import os
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import IterableDataset, DataLoader
from sklearn.metrics import classification_report, precision_recall_fscore_support
from tqdm.auto import tqdm

from datasets import load_dataset
from transformers import (
    DistilBertTokenizerFast,
    DistilBertForSequenceClassification
)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

# Paths
TRAIN_CSV = "/content/drive/MyDrive/Colab Notebooks/fatigue detection/data/oversampling train dataset.csv"
TEST_CSV  = "/content/drive/MyDrive/Colab Notebooks/fatigue detection/data/oversampling test dataset.csv"

# Hyperparameters
BATCH_SIZE = 8
MAX_LEN = 256
LR = 5e-5
NUM_EPOCHS = 1
EARLY_STOPPING_PATIENCE = 2

# Focal Loss config
USE_FOCAL = False
FOCAL_GAMMA = 2.0
FOCAL_ALPHA = torch.tensor([1.0, 3.0, 1.0, 1.0])  # adjust per-class weights here


# ======================================================================
# 1. BUILD LABEL MAP (small memory)
# ======================================================================
import pandas as pd

def build_label_map(path):
    df = pd.read_csv(path)
    labels = sorted(df["label"].unique())
    label2id = {l: i for i, l in enumerate(labels)}
    id2label = {i: l for l, i in label2id.items()}
    print("Label Map:", label2id)
    return label2id, id2label

label2id, id2label = build_label_map(TRAIN_CSV)


# ======================================================================
# 2. STREAMING DATA LOADING (no RAM usage)
# ======================================================================
tokenizer = DistilBertTokenizerFast.from_pretrained("distilbert-base-uncased")

def tokenize_stream(example):
    tok = tokenizer(
        example["clean_text"],
        padding="max_length",
        truncation=True,
        max_length=MAX_LEN
    )
    tok["labels"] = label2id[example["label"]]
    return tok

print("Streaming dataset (no RAM loading)...")
ds_train = load_dataset("csv", data_files=TRAIN_CSV, split="train", streaming=True)
ds_test  = load_dataset("csv", data_files=TEST_CSV,  split="train", streaming=True)

ds_train = ds_train.map(tokenize_stream)
ds_test  = ds_test.map(tokenize_stream)


# ======================================================================
# 3. ITERABLE PYTORCH DATASET WRAPPER
# ======================================================================
class StreamingTorchDataset(IterableDataset):
    def __init__(self, hf_stream_iter):
        self.ds = hf_stream_iter

    def __iter__(self):
        for ex in self.ds:
            yield {
                "input_ids": torch.tensor(ex["input_ids"]),
                "attention_mask": torch.tensor(ex["attention_mask"]),
                "labels": torch.tensor(ex["labels"], dtype=torch.long)
            }

train_ds = StreamingTorchDataset(ds_train)
test_ds  = StreamingTorchDataset(ds_test)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE)
test_loader  = DataLoader(test_ds, batch_size=BATCH_SIZE)


# ======================================================================
# 4. FOCAL LOSS
# ======================================================================
class FocalLoss(nn.Module):
    def __init__(self, gamma=2.0, alpha=None):
        super().__init__()
        self.gamma = gamma
        self.alpha = alpha

    def forward(self, logits, labels):
        ce = F.cross_entropy(logits, labels, reduction="none")
        pt = torch.exp(-ce)
        loss = (1 - pt) ** self.gamma * ce
        if self.alpha is not None:
            alpha = self.alpha.to(labels.device)
            loss = alpha[labels] * loss
        return loss.mean()


# ======================================================================
# 5. LOAD MODEL
# ======================================================================
print(f"Using {'Focal Loss' if USE_FOCAL else 'CrossEntropy'}")

model = DistilBertForSequenceClassification.from_pretrained(
    "distilbert-base-uncased",
    num_labels=len(label2id),
    id2label=id2label,
    label2id=label2id
).to(DEVICE)

criterion = FocalLoss(FOCAL_GAMMA, FOCAL_ALPHA) if USE_FOCAL else nn.CrossEntropyLoss()
optimizer = torch.optim.AdamW(model.parameters(), lr=LR)
scaler = torch.cuda.amp.GradScaler()  # FP16 mixed precision


# ======================================================================
# 6. TRAINING LOOP (Progress Bar + Early Stopping)
# ======================================================================
best_f1 = -1
epochs_no_improve = 0

for epoch in range(1, NUM_EPOCHS+1):
    print(f"\n======== Epoch {epoch}/{NUM_EPOCHS} ========")
    model.train()
    pbar = tqdm(train_loader, desc="Training", leave=True)

    for batch in pbar:
        batch = {k: v.to(DEVICE) for k, v in batch.items()}

        optimizer.zero_grad()
        with torch.cuda.amp.autocast():
            out = model(input_ids=batch["input_ids"], attention_mask=batch["attention_mask"])
            loss = criterion(out.logits, batch["labels"])

        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()

        pbar.set_postfix({"loss": float(loss)})

    # =======================
    # Evaluation
    # =======================
    print("Evaluating…")
    model.eval()
    preds, labels = [], []

    with torch.no_grad():
        for batch in tqdm(test_loader, desc="Testing", leave=True):
            batch = {k: v.to(DEVICE) for k, v in batch.items()}
            with torch.cuda.amp.autocast():
                out = model(batch["input_ids"], batch["attention_mask"])
            preds.extend(out.logits.argmax(dim=1).cpu().numpy())
            labels.extend(batch["labels"].cpu().numpy())

    precision, recall, f1, _ = precision_recall_fscore_support(
        labels, preds, average="weighted", zero_division=0
    )
    print(f"Epoch {epoch} | F1 = {f1:.4f}")

    # Early stopping logic
    if f1 > best_f1:
        best_f1 = f1
        epochs_no_improve = 0
        model.save_pretrained("./best_model")
        tokenizer.save_pretrained("./best_model")
        print("✔ New best model saved!")
    else:
        epochs_no_improve += 1
        if epochs_no_improve >= EARLY_STOPPING_PATIENCE:
            print("⛔ Early stopping triggered!")
            break


# ======================================================================
# 7. FINAL EVALUATION
# ======================================================================
print("\n===== FINAL TEST EVALUATION =====")
model = DistilBertForSequenceClassification.from_pretrained("./best_model").to(DEVICE)

model.eval()
preds, labels = [], []

with torch.no_grad():
    for batch in test_loader:
        batch = {k: v.to(DEVICE) for k, v in batch.items()}
        out = model(batch["input_ids"], batch["attention_mask"])
        preds.extend(out.logits.argmax(dim=1).cpu().numpy())
        labels.extend(batch["labels"].cpu().numpy())

print(classification_report(labels, preds, target_names=list(label2id.keys())))


Label Map: {'Apathy': 0, 'Fatigue': 1, 'Neutral': 2, 'Stress': 3}
Streaming dataset (no RAM loading)...
Using CrossEntropy


Some weights of DistilBertForSequenceClassification were not initialized from the model checkpoint at distilbert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight', 'pre_classifier.bias', 'pre_classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.



======== Epoch 1/1 ========


/tmp/ipython-input-3168621279.py:131: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler()  # FP16 mixed precision


Training: 0it [00:00, ?it/s]

/tmp/ipython-input-3168621279.py:149: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast():
/tmp/ipython-input-3168621279.py:149: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast():
/tmp/ipython-input-3168621279.py:149: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast():
/tmp/ipython-input-3168621279.py:149: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast():
/tmp/ipython-input-3168621279.py:149: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast():
/tmp/ipython-input-316862

Evaluating…


Testing: 0it [00:00, ?it/s]

/tmp/ipython-input-3168621279.py:169: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast():
/tmp/ipython-input-3168621279.py:169: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast():
/tmp/ipython-input-3168621279.py:169: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast():
/tmp/ipython-input-3168621279.py:169: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast():
/tmp/ipython-input-3168621279.py:169: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast():
/tmp/ipython-input-316862

Epoch 1 | F1 = 0.9116
✔ New best model saved!

===== FINAL TEST EVALUATION =====
              precision    recall  f1-score   support

      Apathy       0.97      0.79      0.87     35035
     Fatigue       0.93      0.95      0.94     35034
     Neutral       0.87      0.96      0.91     35035
      Stress       0.89      0.94      0.92     35034

    accuracy                           0.91    140138
   macro avg       0.92      0.91      0.91    140138
weighted avg       0.92      0.91      0.91    140138

